In [71]:
import pandas as pd 
import numpy as np 

df=pd.read_csv('prepared_cric_data.csv')

In [72]:
df.head(150)

,match_id,city,bowling_team,batting_team,balls_left,wickets_left,crr,last_five,current_score2,final_score2
0,1,83,10,0,119,10,0.00,0,0,168
1,1,83,10,0,118,10,0.00,0,0,168
2,1,83,10,0,117,10,2.00,1,1,168
3,1,83,10,0,116,10,4.50,3,3,168
4,1,83,10,0,115,10,3.60,3,3,168
...,...,...,...,...,...,...,...,...,...,...
145,1,83,0,10,101,3,61.26,41,26,172
146,1,83,0,10,100,3,59.40,45,30,172
147,1,83,0,10,99,3,57.71,48,34,172
148,1,83,0,10,98,3,55.36,45,35,172


In [74]:
# Recreate runs_total from current_score
df['runs_total'] = df.groupby(['match_id', 'batting_team'])['current_score2'].diff().fillna(df['current_score2'])

In [75]:
# Fix crr
df['crr'] = (df['current_score2'] / (120 - df['balls_left'])) * 6

# Fix last_five
df['last_five'] = df.groupby(['match_id', 'batting_team'])['runs_total'].rolling(30, min_periods=1).sum().reset_index(level=[0,1], drop=True)

In [77]:
df['crr']=df['crr'].round(2)
df.head(150)

,match_id,city,bowling_team,batting_team,balls_left,wickets_left,crr,last_five,current_score2,final_score2,runs_total
0,1,83,10,0,119,10,0.00,0.0,0,168,0.0
1,1,83,10,0,118,10,0.00,0.0,0,168,0.0
2,1,83,10,0,117,10,2.00,1.0,1,168,1.0
3,1,83,10,0,116,10,4.50,3.0,3,168,2.0
4,1,83,10,0,115,10,3.60,3.0,3,168,0.0
...,...,...,...,...,...,...,...,...,...,...,...
145,1,83,0,10,101,3,8.21,26.0,26,172,4.0
146,1,83,0,10,100,3,9.00,30.0,30,172,4.0
147,1,83,0,10,99,3,9.71,34.0,34,172,4.0
148,1,83,0,10,98,3,9.55,35.0,35,172,1.0


In [78]:
df=df.drop(['runs_total'],axis=1)

In [79]:
df.to_csv('prepared_cric_data.csv',index=False)

In [80]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

for col in df.columns:
    if df[col].dtype=='object'or df[col].dtype=='str':
        df[col]=le.fit_transform(df[col])

In [87]:
from sklearn.model_selection import GroupShuffleSplit

x=df.drop(['final_score2','match_id'],axis=1)
y=df['final_score2']

gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42)

grp=df['match_id']

for tr,ts in gss.split(x,y,grp):
    x_train=x.iloc[tr]
    x_test=x.iloc[ts]
    y_train=y.iloc[tr]
    y_test=y.iloc[ts]


In [88]:
from xgboost import XGBRegressor

model=XGBRegressor(n_estimators=500,learning_rate=0.05,max_depth=8,subsample=0.8,colsample_bytree=0.8,random_state=42)
model.fit(x_train,y_train)
print("-------------- Model Trained --------------")

-------------- Model Trained --------------


In [89]:
from sklearn.metrics import mean_absolute_error,mean_squared_error

y_pred=model.predict(x_test)

mae=mean_absolute_error(y_test,y_pred)
rmse=np.sqrt(mean_squared_error(y_test,y_pred))

print(f"MAE : {mae}")
print(f"RMSE : {rmse}")

MAE : 19.34666633605957
RMSE : 26.50537745778557


In [90]:
import pickle

In [91]:
with open('t20_score_predictor.pkl',"wb")as f:
    pickle.dump(model,f)